# MIF 2026 — estudo final de performance de canais

## tl;dr

Este notebook reexecuta o pipeline de produção sobre as exportações frescas e os mapeamentos revisados. A base final reúne **14.027 pedidos pagos**, **15.713 inscrições pagas** e **R$ 4.321.891,20 de valor bruto**, com 51,56% das inscrições assistidas por cupom. Há **83 canais com dossiê completo** e **73 na cauda compacta**. O estudo descreve perfis complementares; não produz score, ranking ou veredito automático de manutenção/corte.

As datas de venda/inscrição não vieram preenchidas nos participantes, então a série semanal usa a data externa do pedido associado. Ritmo e assessoria/clube também não vieram disponíveis. Patrocínio, espaço de expo, permutas e valor de cortesias permanecem fora do que as exportações mensuram.


## Contexto e método

O pipeline separa os grãos de pedido, inscrição paga e item de produto. Pedidos fornecem os totais financeiros; inscrições sustentam os recortes de canal, modalidade, lote e perfil; itens de produto medem presença e quantidade, sem inferir receita quando não há preço explícito.

As fontes são informadas por variáveis de ambiente e processadas em diretório temporário. O notebook chama os mesmos módulos usados pela CLI, valida o contrato de 32 consultas e persiste somente verificações e agregados anônimos. Índice, dossiês e cauda compacta seguem valor bruto alocado decrescente, inscrições pagas e nome; a auditoria de aliases permanece agrupada por canal e cupom.


In [1]:
from pathlib import Path
from tempfile import TemporaryDirectory
import csv
import json
import os
import sys

project_root = Path(os.environ["MIF_PROJECT_ROOT"]).resolve()
source_dir = Path(os.environ["MIF_SOURCE_DIR"]).resolve()
sys.path.insert(0, str(project_root))

from _codex.analyses.mif_2026_channels.pipeline import run_analysis
from _codex.analyses.mif_2026_channels.run import verify_outputs

analysis_dir = project_root / "_codex/analyses/mif_2026_channels"
workspace = TemporaryDirectory()
output_dir = Path(workspace.name) / "report_app"
outputs = run_analysis(
    source_dir / "orders_72611.csv",
    source_dir / "participants_72611.csv",
    analysis_dir / "channel_mapping.csv",
    analysis_dir / "product_mapping.csv",
    output_dir,
)
verify_outputs(outputs["report_data"], outputs["aggregates"], outputs["reconciliation"], outputs["source_notes"])
report = json.loads(outputs["report_data"].read_text(encoding="utf-8"))
aggregates = json.loads(outputs["aggregates"].read_text(encoding="utf-8"))
reconciliation = json.loads(outputs["reconciliation"].read_text(encoding="utf-8"))
source_notes = json.loads(outputs["source_notes"].read_text(encoding="utf-8"))
assert report["status"] == "ready" and len(report["queries"]) == 32
capstone = report["queries"]["roadrunners_capstone"]["rows"][0]
assert capstone["available"] and capstone["paid_registrations"] == 1876
{"status": report["status"], "queries": len(report["queries"]), "surface": report["surface"], "roadrunners_paid": capstone["paid_registrations"]}


{'status': 'ready', 'queries': 32, 'surface': 'report', 'roadrunners_paid': 1876}

## Qualidade e frescor

Os recibos abaixo identificam as duas extrações sem expor linhas ou identificadores pessoais. A seleção de qualidade concentra os campos que condicionam interpretação geográfica, temporal e comportamental.


In [2]:
source_receipts = [
    {key: row[key] for key in ("source_id", "file_name", "row_count", "sha256", "extracted_at", "event_code")}
    for row in source_notes["sources"]
]
selected_fields = {"order_date", "sale_date", "registration_date", "city", "state", "age", "pace_seconds", "club"}
quality = [
    {key: row[key] for key in ("grain", "field", "valid", "missing", "invalid", "denominator", "coverage_pct")}
    for row in aggregates["datasets"]["data_quality"] if row["field"] in selected_fields
]
{"fresh_receipts": source_receipts, "selected_quality": quality}


{'fresh_receipts': [{'source_id': 'orders', 'file_name': 'orders_72611.csv', 'row_count': 14886, 'sha256': '4d96ecff8174022ba6dade37babf7cc227075f6649371056c92402346a2a0f62', 'extracted_at': '2026-08-31T23:55:57+00:00', 'event_code': 72611}, {'source_id': 'participants', 'file_name': 'participants_72611.csv', 'row_count': 16634, 'sha256': '119d7e1573f2daa114f080283ba64f155884df5fd746eb06ee73d3329b35146a', 'extracted_at': '2026-08-31T23:56:00+00:00', 'event_code': 72611}], 'selected_quality': [{'grain': 'order', 'field': 'order_date', 'valid': 14886, 'missing': 0, 'invalid': 0, 'denominator': 14886, 'coverage_pct': 100.0}, {'grain': 'registration', 'field': 'age', 'valid': 15686, 'missing': 0, 'invalid': 27, 'denominator': 15713, 'coverage_pct': 100.0}, {'grain': 'registration', 'field': 'city', 'valid': 14784, 'missing': 917, 'invalid': 12, 'denominator': 15713, 'coverage_pct': 94.16}, {'grain': 'registration', 'field': 'club', 'valid': 0, 'missing': 15713, 'invalid': 0, 'denominator':

## Reconciliação e cobertura

Modalidade, lote, canal e série semanal precisam fechar exatamente com o total de inscrições pagas. Coberturas de junção e mapeamento devem ser integrais antes de usar os perfis.


In [3]:
overview = aggregates["overview"]
datasets = aggregates["datasets"]
checks = {
    "receipt_vs_overview": reconciliation["paid_registration_count"] == overview["paid_registrations"],
    "modalities_vs_overview": sum(row["paid_registrations"] for row in datasets["modality_mix"]) == overview["paid_registrations"],
    "lots_vs_overview": sum(row["paid_registrations"] for row in datasets["lot_performance"]) == overview["paid_registrations"],
    "channels_vs_overview": sum(row["paid_registrations"] for row in datasets["channel_index"]) == overview["paid_registrations"],
    "weeks_vs_overview": sum(row["paid_registrations"] for row in datasets["weekly_sales"]) == overview["paid_registrations"],
}
coverage = {
    "registration_join_coverage_pct": reconciliation["registration_join_coverage_pct"],
    "channel_mapping_coverage_pct": reconciliation["channel_mapping_coverage_pct"],
    "product_mapping_coverage_pct": reconciliation["product_mapping_coverage_pct"],
}
assert all(checks.values()) and set(coverage.values()) == {100.0}
{"additive_checks": checks, "coverage": coverage}


{'additive_checks': {'receipt_vs_overview': True, 'modalities_vs_overview': True, 'lots_vs_overview': True, 'channels_vs_overview': True, 'weeks_vs_overview': True}, 'coverage': {'registration_join_coverage_pct': 100.0, 'channel_mapping_coverage_pct': 100.0, 'product_mapping_coverage_pct': 100.0}}

## Manchetes e mapeamentos revisados

O recorte a seguir mantém os quatro canais especiais pedidos para leitura explícita. São descrições observadas, não uma decisão comercial automática.


In [4]:
with (analysis_dir / "channel_mapping.csv").open(encoding="utf-8", newline="") as handle:
    channel_rows = list(csv.DictReader(handle))
with (analysis_dir / "product_mapping.csv").open(encoding="utf-8", newline="") as handle:
    product_rows = list(csv.DictReader(handle))

special_names = {"Corre Criciúma", "Sports Week", "PCD", "Benefício"}
special_channels = [
    {key: row[key] for key in ("channel_name", "channel_type", "paid_registrations", "touched_paid_orders", "gross_value", "registration_ticket", "dossier_type")}
    for row in datasets["channel_index"] if row["channel_name"] in special_names
]
product_class_totals = {
    label: sum(row["product_quantity"] for row in datasets["product_summary"] if row["classification"] == label)
    for label in ("kit_incluso", "adicional")
}
headline = {
    "paid_orders": overview["paid_orders"],
    "paid_registrations": overview["paid_registrations"],
    "gross_value": overview["gross_value"],
    "discount_value": overview["discount_value"],
    "fee_value": overview["fee_value"],
    "net_transfer_value": overview["net_transfer_value"],
    "order_ticket": overview["order_ticket"],
    "registration_ticket": overview["registration_ticket"],
    "coupon_assisted_share_pct": overview["coupon_assisted_share_pct"],
    "full_dossiers": len(aggregates["full_dossiers"]),
    "compact_channels": len(aggregates["long_tail"]),
}
mapping_summary = {
    "coupon_pairs": len(channel_rows),
    "mapped_channels_excluding_organic": len({row["channel_name"] for row in channel_rows}),
    "product_identities": len(product_rows),
    "canonical_products": len({row["canonical_name"] for row in product_rows}),
    "product_class_totals": product_class_totals,
    "explicit_product_revenue_available": any(row["explicit_revenue"] is not None for row in datasets["product_summary"]),
}
assert len(special_channels) == 4
{"headline": headline, "mapping": mapping_summary, "special_channels": special_channels}


{'headline': {'paid_orders': 14027, 'paid_registrations': 15713, 'gross_value': '4321891.20', 'discount_value': '530601.25', 'fee_value': '372816.27', 'net_transfer_value': '4024811.72', 'order_ticket': '308.11', 'registration_ticket': '275.05', 'coupon_assisted_share_pct': 51.56, 'full_dossiers': 83, 'compact_channels': 73}, 'mapping': {'coupon_pairs': 1002, 'mapped_channels_excluding_organic': 155, 'product_identities': 33, 'canonical_products': 20, 'product_class_totals': {'kit_incluso': 15945, 'adicional': 3290}, 'explicit_product_revenue_available': False}, 'special_channels': [{'channel_name': 'Sports Week', 'channel_type': 'evento_acao', 'paid_registrations': 949, 'touched_paid_orders': 841, 'gross_value': '233215.78', 'registration_ticket': '245.75', 'dossier_type': 'full'}, {'channel_name': 'PCD', 'channel_type': 'politica', 'paid_registrations': 71, 'touched_paid_orders': 71, 'gross_value': '9463.07', 'registration_ticket': '133.28', 'dossier_type': 'full'}, {'channel_name': 

## Fronteira de privacidade

Somente os quatro artefatos anônimos temporários são examinados. O teste impede a presença de chaves de identificação e de marcadores das fixtures de desenvolvimento.


In [5]:
combined = "\n".join(path.read_text(encoding="utf-8") for path in outputs.values()).lower()
forbidden_tokens = [
    "numero" + "_pedido", "numero" + "_inscricao", '"' + 'facts' + '"',
    "example" + ".test", "segredo" + "-teste", "base" + " sintética",
    "task " + "8 fará", "serve apenas" + " para provar",
]
privacy_checks = {token: token not in combined for token in forbidden_tokens}
assert all(privacy_checks.values())
{"checked_artifacts": len(outputs), "forbidden_markers_found": sum(not ok for ok in privacy_checks.values())}


{'checked_artifacts': 4, 'forbidden_markers_found': 0}

## Conclusões de uso

- O snapshot final está em `ready`, com 32 consultas e recibos frescos das duas fontes.
- Pedidos, inscrições, canais, modalidades, lotes e semanas reconciliam; junção e mapeamentos têm cobertura de 100%.
- Os 1.002 pares exatos de cupom consolidam 155 canais mapeados; orgânico é apresentado separadamente no relatório.
- Entre as inscrições pagas, os produtos consolidados somam 15.945 itens de kit e 3.290 adicionais, sem receita de produto inferida.
- ROADRUNNERS reúne 1.876 inscrições pagas (11,94% do evento) e o capstone preserva separadamente escala, alcance, composição e semelhanças.
- A ordem comercial de leitura usa valor bruto alocado, inscrições pagas e nome, sem alterar a auditoria completa de aliases nem criar avaliação de qualidade.
- A escolha de canais para 2027 deve combinar perfis diferentes e considerar evidências externas que esta exportação não contém; o relatório não automatiza essa decisão.
